# Financial Fraud Detection with IBM Watsonx Governance

This notebook demonstrates the end-to-end fraud detection pipeline:

1. **Synthetic Data Generation** - Generate realistic transaction data
2. **Feature Engineering** - Extract temporal, velocity, amount, and geo features
3. **Model Training** - Train the ensemble (XGBoost + LightGBM + Isolation Forest)
4. **Prediction with Explainability** - SHAP explanations for each prediction
5. **Governance Factsheet** - EU AI Act Article 11 compliance documentation

---

## Setup

Ensure you have installed the project dependencies:
```bash
pip install -r requirements.txt
```

In [ ]:
import os
import sys
import warnings

# Add project root to path
sys.path.insert(0, os.path.abspath(".."))
os.environ["PYTHONPATH"] = os.path.abspath("..")

warnings.filterwarnings("ignore")

import json

import numpy as np

print("Environment ready.")

## 1. Synthetic Data Generation

Generate realistic financial transaction data with configurable fraud rate.
The generator creates transactions with:
- Temporal patterns (timestamps over 90 days)
- Legitimate amounts (lognormal ~$33 median) vs. fraudulent amounts (lognormal ~$245 median)
- Protected attributes (age group, gender, geography) for bias testing

In [ ]:
from src.data.synthetic_generator import SyntheticTransactionGenerator

config = {
    "n_transactions": 5000,
    "fraud_rate": 0.03,
    "n_merchants": 100,
    "n_customers": 500,
    "seed": 42,
}

generator = SyntheticTransactionGenerator(config=config)
df = generator.generate()

print(f"Total transactions: {len(df):,}")
print(f"Fraud transactions: {df['is_fraud'].sum():,.0f} ({df['is_fraud'].mean():.1%})")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
# Distribution of amounts: legitimate vs. fraudulent
print("--- Legitimate Transactions ---")
print(df[df["is_fraud"] == 0]["amount"].describe().round(2))
print("\n--- Fraudulent Transactions ---")
print(df[df["is_fraud"] == 1]["amount"].describe().round(2))

In [ ]:
# Protected attribute distributions
print("Age Group Distribution:")
print(df["customer_age_group"].value_counts())
print("\nGender Distribution:")
print(df["customer_gender"].value_counts())
print("\nGeography Distribution:")
print(df["customer_geo"].value_counts())

## 2. Feature Engineering

The `FeatureEngineer` extracts the following feature groups:

| Group | Features |
|---|---|
| **Temporal** | hour, day_of_week, is_weekend, is_night, cyclical sin/cos |
| **Velocity** | Transaction count/sum/mean/std over 1h, 6h, 24h, 72h windows |
| **Amount** | Customer mean/std/min/max, z-score, max ratio, log amount |
| **Merchant** | Merchant avg/std amount, txn count, unique merchants, is_new |
| **Geographic** | Haversine distance from last transaction, geo anomaly flag |

In [ ]:
from src.data.feature_engineering import FeatureEngineer

# Use smaller velocity windows for notebook speed
fe_config = {
    "velocity": {"windows": [1, 6]},
    "amount": {"percentiles": [25, 50, 75, 90, 95, 99]},
    "temporal": {"cyclical": True},
    "geo": {"max_distance_km": 500.0},
}

fe = FeatureEngineer(config=fe_config)
df_features = fe.transform(df)

feature_names = fe.get_feature_names()
print(f"Engineered features: {len(feature_names)}")
print(f"Feature names: {feature_names}")

In [ ]:
# Prepare feature matrix for modeling
available_features = [f for f in feature_names if f in df_features.columns]
X = df_features[available_features].values.astype(np.float64)
y = df_features["is_fraud"].values.astype(int)

# Handle any NaN values
X = np.nan_to_num(X, nan=0.0)

print(f"Feature matrix shape: {X.shape}")
print(f"Label distribution: {dict(zip(*np.unique(y, return_counts=True), strict=False))}")

## 3. Model Training

Train the weighted ensemble model:
- **XGBoost** (40% weight): Gradient boosted trees optimized for AUC-PR
- **LightGBM** (40% weight): Fast gradient boosting with leaf-wise growth
- **Isolation Forest** (20% weight): Unsupervised anomaly detection

The trainer performs stratified k-fold cross-validation before fitting on full data.

In [ ]:
from src.models.trainer import FraudModelTrainer

# Use small models for notebook speed
ensemble_config = {
    "weights": {"xgboost": 0.40, "lightgbm": 0.40, "isolation_forest": 0.20},
    "threshold": {"fraud": 0.70, "review": 0.40},
    "xgboost": {
        "n_estimators": 50,
        "max_depth": 5,
        "learning_rate": 0.1,
        "scale_pos_weight": 10,
        "random_state": 42,
    },
    "lightgbm": {
        "n_estimators": 50,
        "max_depth": 5,
        "learning_rate": 0.1,
        "scale_pos_weight": 10,
        "random_state": 42,
    },
    "isolation_forest": {
        "n_estimators": 50,
        "contamination": 0.03,
        "random_state": 42,
    },
}

trainer = FraudModelTrainer(n_folds=3, ensemble_config=ensemble_config)
model = trainer.train(X, y)

print("\n--- Cross-Validation Metrics ---")
metrics = trainer.get_metrics()
for key, value in metrics.items():
    if key != "fold_scores":
        print(f"  {key}: {value}")

In [ ]:
# Generate predictions
transaction_ids = df_features["transaction_id"].tolist()
predictions = model.predict(X, transaction_ids=transaction_ids)

# Summary
labels = [p.label for p in predictions]
print("Prediction Distribution:")
for label in ["fraud", "review", "legitimate"]:
    count = labels.count(label)
    print(f"  {label}: {count} ({count / len(labels):.1%})")

print("\nSample fraud prediction:")
fraud_preds = [p for p in predictions if p.label == "fraud"]
if fraud_preds:
    p = fraud_preds[0]
    print(f"  Transaction: {p.transaction_id}")
    print(f"  Probability: {p.fraud_probability:.4f}")
    print(f"  Model Scores: {p.model_scores}")

## 4. Prediction with SHAP Explainability

Generate SHAP explanations for individual predictions.
The `ShapExplainer` uses `TreeExplainer` for XGBoost and LightGBM to provide
exact, fast SHAP values showing each feature's contribution to the fraud score.

In [ ]:
from src.explainability.shap_explainer import ShapExplainer

explainer = ShapExplainer(top_k=5)
explainer.fit(model)

# Explain a flagged transaction
if fraud_preds:
    idx = transaction_ids.index(fraud_preds[0].transaction_id)
    explanation = explainer.explain_single(
        X[idx : idx + 1],
        available_features,
        fraud_preds[0].transaction_id,
    )

    print(f"SHAP Explanation for {explanation.transaction_id}")
    print(f"Base value: {explanation.base_value:.4f}")
    print("\nTop Risk Factors (increasing fraud score):")
    for name, val in explanation.top_positive_features:
        print(f"  {name}: {val:+.4f}")
    print("\nTop Mitigating Factors (decreasing fraud score):")
    for name, val in explanation.top_negative_features:
        print(f"  {name}: {val:+.4f}")

In [ ]:
# Generate narrative explanation (uses mock when Watsonx API key is not set)
from src.explainability.narrative_generator import NarrativeGenerator

narrator = NarrativeGenerator()

if fraud_preds and explanation:
    narrative = narrator.generate(
        explanation=explanation,
        fraud_probability=fraud_preds[0].fraud_probability,
        label=fraud_preds[0].label,
    )
    print("Automated Narrative:")
    print(narrative)

## 5. Governance: Bias Detection

Evaluate model fairness across protected attributes (age group, gender, geography)
using demographic parity and equalized odds metrics.

In [ ]:
from src.governance.bias_detector import BiasDetector

bias_config = {
    "demographic_parity_threshold": 0.10,
    "equalized_odds_threshold": 0.10,
    "protected_attributes": ["customer_age_group", "customer_gender", "customer_geo"],
}

bias_detector = BiasDetector(config=bias_config)
y_pred = np.array([1 if p.label == "fraud" else 0 for p in predictions])

bias_report = bias_detector.evaluate(
    y_true=y,
    y_pred=y_pred,
    protected_df=df_features,
)

print(f"Is Fair: {bias_report['is_fair']}")
print(f"Overall Positive Rate: {bias_report['overall_positive_rate']:.4f}")
print("\nAlerts:")
for alert in bias_report["alerts"]:
    print(f"  - {alert}")
if not bias_report["alerts"]:
    print("  (no bias alerts)")

## 6. Governance: Drift Monitoring

Monitor data distribution drift using Population Stability Index (PSI)
and Kolmogorov-Smirnov test.

In [ ]:
from src.governance.drift_monitor import DriftMonitor

drift_monitor = DriftMonitor(config={"psi_threshold": 0.20, "ks_threshold": 0.05})

# Simulate reference vs. production split
split = int(len(X) * 0.7)
X_ref = X[:split]
X_prod = X[split:]

drift_report = drift_monitor.evaluate_drift(X_ref, X_prod, available_features)

print(f"Drift Detected: {drift_report['drift_detected']}")
print(
    f"Drifted Features: {drift_report['drifted_features_count']} / {drift_report['total_features']}"
)
if drift_report["drifted_features"]:
    print(f"Features with drift: {drift_report['drifted_features']}")

## 7. EU AI Act Article 11 Factsheet

Generate a comprehensive model factsheet compliant with EU AI Act Article 11,
including all governance metrics, bias assessments, and drift reports.

In [ ]:
from src.governance.factsheet import ModelFactsheet

factsheet_gen = ModelFactsheet()

training_data_summary = {
    "description": "Synthetic financial transaction data",
    "size": len(df),
    "fraud_rate": f"{df['is_fraud'].mean():.2%}",
    "date_range": f"{df['timestamp'].min()} to {df['timestamp'].max()}",
    "n_customers": df["customer_id"].nunique(),
    "n_merchants": df["merchant_id"].nunique(),
}

factsheet = factsheet_gen.generate(
    model_metrics=metrics,
    bias_report=bias_report,
    drift_report=drift_report,
    training_data_summary=training_data_summary,
)

print(json.dumps(factsheet, indent=2, default=str)[:3000])
print("\n... (truncated for display)")

## Summary

This notebook demonstrated the complete fraud detection pipeline:

1. Generated synthetic transaction data with configurable fraud patterns
2. Engineered temporal, velocity, amount, merchant, and geographic features
3. Trained a weighted ensemble (XGBoost + LightGBM + Isolation Forest)
4. Generated SHAP explanations and natural-language narratives for flagged transactions
5. Evaluated bias across protected attributes (age, gender, geography)
6. Monitored data drift using PSI and KS-test
7. Generated an EU AI Act Article 11 compliant model factsheet

For production deployment, connect the IBM Watsonx API key for Granite LLM narratives
and configure Kafka for real-time transaction streaming.